# Module 1: Prompt Engineering

**Goal:** Master the core prompting techniques that form the foundation of all LLM work.

**What you'll learn:**
- Zero-shot vs. few-shot prompting
- Chain-of-thought (CoT) reasoning
- Role prompting
- Structured output (JSON extraction)
- Iterative prompt refinement

**Prerequisites:** Set up your `.env` file with an API key. See `SETUP.md` at the repo root.

---

## 0. Setup

In [ ]:
# Install dependencies for this module
%pip install -q openai anthropic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load .env from repo root
load_dotenv(dotenv_path="../.env")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def chat(prompt: str, model: str = "gpt-4o-mini", system: str = None) -> str:
    """Helper: send a prompt, return the response text."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7,
    )
    return response.choices[0].message.content

print("✅ Setup complete")

---
## 1. Zero-Shot Prompting

Ask the model to perform a task **with no examples**. Works well for tasks the model was trained on extensively.

In [ ]:
# Zero-shot: direct instruction, no examples
prompt = "Classify the sentiment of this review as Positive, Negative, or Neutral:\n\n'The product quality is acceptable, nothing special.'"

response = chat(prompt)
print("Response:", response)

In [ ]:
# 🧪 EXERCISE: Try your own zero-shot prompt
# Ideas: translation, summarization, code explanation, tone detection

my_prompt = """TODO: write your prompt here"""

# response = chat(my_prompt)
# print(response)

---
## 2. Few-Shot Prompting

Provide 2–5 examples of input→output pairs **inside the prompt** to teach the model the pattern. 

**Why it works:** The model infers the task format from examples rather than relying solely on instruction wording.

In [ ]:
few_shot_prompt = """Convert informal English to formal English.

Input: "Hey, what's up?"
Output: "Greetings, how are you doing?"

Input: "Thanks a lot!"
Output: "Thank you very much."

Input: "Can you help me out?"
Output: "Could you assist me, please?"

Input: "I wanna grab coffee later."
Output:"""

response = chat(few_shot_prompt)
print("Formal version:", response)

In [ ]:
# 🧪 EXERCISE: Vary the number of examples (1-shot vs 3-shot vs 5-shot)
# Observe how response consistency changes.
# Try a completely different task: code style conversion, tone matching, etc.

---
## 3. Chain-of-Thought (CoT) Prompting

Instruct the model to **think step by step** before answering. Dramatically improves accuracy on reasoning, math, and logic tasks.

**Key insight:** The model's "scratch pad" (the reasoning steps) allows it to maintain intermediate state, which the context window alone can't do implicitly.

In [ ]:
# Without CoT
no_cot = """A store sells apples for $0.50 each and oranges for $0.75 each. 
If I buy 6 apples and 4 oranges, and pay with a $10 bill, how much change do I get?"""

print("Without CoT:")
print(chat(no_cot))
print()

In [ ]:
# With CoT — same question, but ask for step-by-step reasoning
with_cot = """A store sells apples for $0.50 each and oranges for $0.75 each. 
If I buy 6 apples and 4 oranges, and pay with a $10 bill, how much change do I get?

Let's think step by step:"""

print("With CoT:")
print(chat(with_cot))

In [ ]:
# 🧪 EXERCISE: Test CoT on a logic puzzle or multi-step coding problem.
# Does it make a difference? When does CoT NOT help?

---
## 4. Role Prompting (System Prompts)

Assign a **persona or role** via the system message to steer tone, depth, and vocabulary. The system prompt persists across all turns in a conversation.

In [ ]:
question = "Explain how neural networks learn."

roles = {
    "ELI5": "You are a teacher explaining concepts to a curious 8-year-old. Use simple words and fun analogies.",
    "Expert": "You are a senior ML researcher. Be precise and technical. Assume the reader has a PhD in CS.",
    "Socratic": "You are a Socratic tutor. Instead of explaining, ask 3 probing questions that guide the learner to discover the answer themselves.",
}

for role_name, system_prompt in roles.items():
    print(f"\n{'='*50}")
    print(f"Role: {role_name}")
    print('='*50)
    response = chat(question, system=system_prompt)
    print(response[:400], "...")  # Truncate for display

In [ ]:
# 🧪 EXERCISE: Design a role for your specific use case.
# Examples: Customer support agent, SQL tutor, code reviewer, product manager.
# Notice how the SAME question gets completely different answers.

---
## 5. Structured Output (JSON Extraction)

Force the model to return machine-parseable output — critical for production systems where you need to process the response programmatically.

In [ ]:
import json

review_text = """
I bought this laptop last month and I'm mostly satisfied. The battery life is fantastic — 
easily 12 hours. However, the keyboard feel is a bit mushy and the fan gets loud under load. 
Overall, good value for $899.
"""

extraction_prompt = f"""Extract structured information from this product review.
Return ONLY valid JSON with these exact fields:
{{
  "sentiment": "positive|negative|neutral|mixed",
  "rating_implied": <1-5 integer>,
  "pros": ["..."],
  "cons": ["..."],
  "price_mentioned": <number or null>,
  "would_recommend": <true|false|null>
}}

Review:
{review_text}"""

raw_response = chat(extraction_prompt, model="gpt-4o-mini")
print("Raw response:")
print(raw_response)

# Parse as JSON
try:
    # Strip markdown code fences if present
    clean = raw_response.strip().removeprefix("```json").removeprefix("```").removesuffix("```")
    parsed = json.loads(clean)
    print("\n✅ Parsed successfully:")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"\n❌ JSON parse error: {e}")

In [ ]:
# Better approach: use OpenAI's response_format=json_object for guaranteed JSON

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a data extractor. Always respond with valid JSON only."},
        {"role": "user", "content": extraction_prompt}
    ],
    response_format={"type": "json_object"},  # Guaranteed JSON
    temperature=0,  # Deterministic for extraction
)

result = json.loads(response.choices[0].message.content)
print("✅ Guaranteed JSON output:")
print(json.dumps(result, indent=2))

---
## 6. Iterative Prompt Refinement

Real prompts are never perfect on the first try. This section demonstrates a structured refinement workflow.

In [ ]:
topic = "quantum computing"

# v1: Vague instruction
v1 = f"Explain {topic}."

# v2: Add constraints
v2 = f"Explain {topic} in exactly 3 sentences. Use an analogy in the first sentence."

# v3: Add audience + format
v3 = f"""Explain {topic} to a software engineer who knows classical computing but not quantum.
Structure:
1. One-sentence analogy to classical computing
2. The key difference (superposition/entanglement)
3. One concrete use case they'd care about

Keep it under 100 words."""

for i, prompt in enumerate([v1, v2, v3], 1):
    print(f"\n{'─'*50}")
    print(f"Prompt v{i}:")
    print(f"{prompt[:80]}{'...' if len(prompt) > 80 else ''}")
    print(f"\nResponse v{i}:")
    print(chat(prompt))

---
## 7. Prompt Engineering Best Practices Cheat Sheet

| Technique | When to use | Key tip |
|-----------|-------------|--------|
| **Zero-shot** | Common tasks (sentiment, translation) | Clear, specific instruction |
| **Few-shot** | Unusual formats or style matching | 3–5 diverse examples |
| **CoT** | Math, logic, multi-step reasoning | "Let's think step by step" |
| **Role prompting** | Tone, expertise, persona control | System message, not user message |
| **Structured output** | Production code, data pipelines | Use `response_format=json_object` |
| **Delimiters** | Long inputs with many parts | `"""`, `---`, `<tag>` |
| **Temperature=0** | Extraction, classification | Deterministic output |
| **Temperature=0.7–1.0** | Creative writing, brainstorming | More varied output |

### The Iterative Improvement Loop

```
Write prompt → Test on 5–10 inputs → Identify failures → Add constraints → Repeat
```

---
**Next:** [Module 02 — RAG Systems](../02-rag-systems/rag_systems.ipynb)

---
## 🧪 Exercises

1. **Few-Shot Modification**: Change the few-shot examples and observe how output style changes.

2. **Role Comparison**: Add a `role` parameter and compare responses (e.g., "pirate" vs "professor").

3. **JSON Schema**: Change the extraction schema to extract different fields from the same text.

4. **CoT Challenge**: Build a chain-of-thought prompt for a math word problem. Compare with zero-shot.